In [1]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 設定路徑與目標工作表
inDir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\02_新竹分署_113年樣區資料_交署\新竹分署_113年樣區資料_交署\新竹分署_113年永久樣區資料_31個"
sheet_name = '樣木'

# 設定資料庫儲存路徑（將資料庫放在與資料夾同層或自訂位置）
db_path = Path(inDir).parent / "forest_plots.db"

root_dir = Path(inDir)
excel_files = root_dir.rglob("*.xls*")

# 建立或連接到 SQLite 資料庫
print(f"正在建立/連接資料庫：{db_path}")
conn = sqlite3.connect(db_path)

# 用來記錄成功匯入與失敗的計數器
success_count = 0
failed_count = 0

for file_path in excel_files:
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} from {file_path.parent}...")
    df = None  # 預設為 None
    
    try:
        # 1. 檢查檔案開頭是否為 XML
        is_xml_spreadsheet = False
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        # 2. 根據真實格式讀取
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name and ws_name[0] == sheet_name:
                    target_worksheet = ws
                    break
            
            if target_worksheet is not None:
                rows_data = []
                for row in target_worksheet.findall('.//Row'):
                    row_cells = []
                    for cell in row.findall('.//Data'):
                        cell_text = cell.text if cell.text else ""
                        row_cells.append(cell_text)
                    if row_cells:
                        rows_data.append(row_cells)
                
                if rows_data:
                    raw_df = pd.DataFrame(rows_data)
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
        else:
            if file_path.suffix.lower() == '.xls':
                df = pd.read_excel(file_path, sheet_name=sheet_name, engine='xlrd')
            else:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')
            
            # 【重要】新增追溯欄位：記錄這筆樣木資料來自哪個檔案與樣區
            # 假設檔案名稱或上層資料夾包含樣區代號，這能幫你做後續的 SQL 篩選
            df['source_file'] = file_path.name
            df['folder_name'] = file_path.parent.name
            
            # 將欄位名稱與資料內容全轉為字串，避免不同檔案間同欄位型態不一致導致寫入失敗
            df = df.astype(str)
            
            # 將資料寫入 SQLite。 
            # name='sample_trees' 是資料庫中的表名
            # if_exists='append' 表示如果表已經存在，就把資料接到後面
            # index=False 表示不要把 Pandas 的流水號索引當成獨立欄位存進去
            df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            
            print(f"成功將 {len(df)} 筆樣木資料匯入資料庫。")
            success_count += 1
        else:
            print(f"⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        print(f"❌ 讀取或寫入資料庫時發生錯誤: {e}")
        failed_count += 1
        
    print('-' * 50)

# 關閉資料庫連線
conn.close()

print("\n匯入任務結束！")
print(f"總共成功匯入 {success_count} 個檔案，失敗 {failed_count} 個檔案。")
print(f"資料庫檔案路徑: {db_path}")

正在建立/連接資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\02_新竹分署_113年樣區資料_交署\新竹分署_113年樣區資料_交署\forest_plots.db
Reading p_06002.xls from C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\02_新竹分署_113年樣區資料_交署\新竹分署_113年樣區資料_交署\新竹分署_113年永久樣區資料_31個...
成功將 46 筆樣木資料匯入資料庫。
--------------------------------------------------
Reading p_06006.xls from C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\02_新竹分署_113年樣區資料_交署\新竹分署_113年樣區資料_交署\新竹分署_113年永久樣區資料_31個...
成功將 51 筆樣木資料匯入資料庫。
--------------------------------------------------
Reading p_06015.xls from C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\02_新竹分署_113年樣區資料_交署\新竹分署_113年樣區資料_交署\新竹分署_113年永久樣區資料_31個...
成功將 33 筆樣木資料匯入資料庫。
--------------------------------------------------
Reading p_06019.xls from C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\02_新竹分署_113年樣區資料_交署\新竹分署_113年樣區資料_交署\新竹分署_113年永久樣區資料_31個...
成功將 32 筆樣木資料匯入資料庫。
--------------------------------------------------
Reading p_06023.xls from C:\Users\a0909\OneDrive\De

In [2]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 🎯 這裡只需要提供最外層的大資料夾路徑即可！
main_dir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料"
sheet_name = '樣木'

# 設定總資料庫儲存路徑（直接放在大資料夾底下）
root_dir = Path(main_dir)
db_path = root_dir / "all_forest_plots_auto.db"

# 建立或連接到 SQLite 資料庫
print(f"正在建立/連接總資料庫：{db_path}")
conn = sqlite3.connect(db_path)

# 用來記錄全局成功與失敗的計數器
total_success_count = 0
total_failed_count = 0

# 🚀 使用 rglob("*.xls*") 直接對最外層大資料夾進行深度搜尋，它會自動掘進所有子目錄
print("正在自動掃描大資料夾下所有的 Excel 檔案...")
excel_files = root_dir.rglob("*.xls*")

for file_path in excel_files:
    # 跳過 Excel 的暫存檔
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} ...")
    df = None
    
    try:
        # 1. 檢查檔案開頭是否為 XML
        is_xml_spreadsheet = False
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        # 2. 根據真實格式讀取
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name and ws_name[0] == sheet_name:
                    target_worksheet = ws
                    break
            
            if target_worksheet is not None:
                rows_data = []
                for row in target_worksheet.findall('.//Row'):
                    row_cells = []
                    for cell in row.findall('.//Data'):
                        cell_text = cell.text if cell.text else ""
                        row_cells.append(cell_text)
                    if row_cells:
                        rows_data.append(row_cells)
                
                if rows_data:
                    raw_df = pd.DataFrame(rows_data)
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
        else:
            if file_path.suffix.lower() == '.xls':
                df = pd.read_excel(file_path, sheet_name=sheet_name, engine='xlrd')
            else:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')
            
            # 💡 【自動智慧欄位解析】
            # 利用 file_path.parts 來取得檔案相對於大資料夾的路徑層級
            # file_path.parent.name 通常是個別的「樣區編號」或「公司代號」資料夾
            # 我們可以抓取上一層或上兩層的目錄名稱，來自動識別所屬的分署與樣區類型
            df['source_file'] = file_path.name           # 檔案名稱
            df['parent_folder'] = file_path.parent.name   # 直接上層資料夾名稱
            
            # 自動抓取路徑中代表「新竹永久_31個」或「宜蘭系統_34個」那一層的目錄名
            # 這裡提供一個安全的寫法：紀錄檔案完整的相對路徑，這樣絕對不會漏掉資訊
            df['relative_path'] = str(file_path.relative_to(root_dir))
            
            # 欄位全轉為字串避免型態衝突
            df = df.astype(str)
            
            # 寫入總資料庫的同一張表 'sample_trees'
            df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            
            print(f"  └ 成功匯入 {len(df)} 筆樣木資料。")
            total_success_count += 1
        else:
            print(f"  ⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        print(f"  ❌ 讀取或寫入資料庫時發生錯誤: {e}")
        total_failed_count += 1
        
    print('-' * 50)

# 關閉資料庫連線
conn.close()

print("\n============================================================")
print("🎉 大資料夾全自動掃描匯入結束！")
print(f"總共成功匯入 {total_success_count} 個檔案，失敗 {total_failed_count} 個檔案。")
print(f"總資料庫檔案路徑: {db_path}")
print("============================================================")

正在建立/連接總資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\all_forest_plots_auto.db
正在自動掃描大資料夾下所有的 Excel 檔案...
Reading 113系統及永久樣區複查數量表(79個)_資料繳交統計.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Worksheet named '樣木' not found
--------------------------------------------------
Reading p_06002.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06006.xls ...
  └ 成功匯入 51 筆樣木資料。
--------------------------------------------------
Reading p_06015.xls ...
  └ 成功匯入 33 筆樣木資料。
--------------------------------------------------
Reading p_06019.xls ...
  └ 成功匯入 32 筆樣木資料。
--------------------------------------------------
Reading p_06023.xls ...
  └ 成功匯入 29 筆樣木資料。
--------------------------------------------------
Reading p_06026.xls ...
  └ 成功匯入 56 筆樣木資料。
--------------------------------------------------
Reading p_06028.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06030.xls ...
  └ 成功匯入 33 筆樣木資料。
-------------------------------------

In [3]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 🎯 這裡只需要提供最外層的大資料夾路徑即可！
main_dir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料"
sheet_name = '樣木'

# 設定總資料庫儲存路徑（直接放在大資料夾底下）
root_dir = Path(main_dir)
db_path = root_dir / "total_forest_plots_auto.db"

# 建立或連接到 SQLite 資料庫
print(f"正在建立/連接總資料庫：{db_path}")
conn = sqlite3.connect(db_path)

# 用來記錄全局成功與失敗的計數器
total_success_count = 0
total_failed_count = 0

# 🚀 使用 rglob("*.xls*") 直接對最外層大資料夾進行深度搜尋，它會自動掘進所有子目錄
print("正在自動掃描大資料夾下所有的 Excel 檔案...")
excel_files = root_dir.rglob("*.xls*")

for file_path in excel_files:
    # 跳過 Excel 的暫存檔
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} ...")
    df = None
    
    try:
        # 1. 檢查檔案開頭是否為 XML
        is_xml_spreadsheet = False
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        # 2. 根據真實格式讀取
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name and ws_name[0] == sheet_name:
                    target_worksheet = ws
                    break
            
            if target_worksheet is not None:
                rows_data = []  # 這裡改用來儲存每一列的字典 (dict)
                
                for row in target_worksheet.findall('.//Row'):
                    row_dict = {}
                    current_col = 1  # Excel XML 的欄位索引預設從 1 開始
                    
                    # 改為先尋找 Cell 標籤，而不是直接找 Data
                    for cell in row.findall('Cell'):
                        # 檢查 Cell 是否有 Index 屬性（用來處理跳格、空格）
                        idx_attr = [v for k, v in cell.attrib.items() if 'Index' in k]
                        if idx_attr:
                            current_col = int(idx_attr[0])  # 如果有指定 Index，強制跳到該欄位序號
                        
                        # 再從 Cell 之中尋找 Data 標籤
                        data_elem = cell.find('Data')
                        cell_text = data_elem.text if (data_elem is not None and data_elem.text) else ""
                        
                        # 將資料根據準確的欄位序號存入字典
                        row_dict[current_col] = cell_text
                        current_col += 1  # 預設自動往下一個欄位移動
                    
                    if row_dict:
                        rows_data.append(row_dict)
                
                if rows_data:
                    # 將字典列表轉為 DataFrame，Pandas 會非常聰明地根據 key (欄位序號) 自動對齊！
                    raw_df = pd.DataFrame(rows_data)
                    
                    # 補齊可能因末尾欄位缺失而沒產生出來的行數，並確保欄位順序從 1 開始連續排列
                    all_cols = range(1, int(raw_df.columns.max()) + 1)
                    raw_df = raw_df.reindex(columns=all_cols).fillna("")
                    
                    # 取第一列作為表頭，並清理表頭的換行符號
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
                    
        else:
            if file_path.suffix.lower() == '.xls':
                df = pd.read_excel(file_path, sheet_name=sheet_name, engine='xlrd')
            else:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')
            
            # 💡 【自動智慧欄位解析】
            # 利用 file_path.parts 來取得檔案相對於大資料夾的路徑層級
            # file_path.parent.name 通常是個別的「樣區編號」或「公司代號」資料夾
            # 我們可以抓取上一層或上兩層的目錄名稱，來自動識別所屬的分署與樣區類型
            df['source_file'] = file_path.name           # 檔案名稱
            df['parent_folder'] = file_path.parent.name   # 直接上層資料夾名稱
            
            # 自動抓取路徑中代表「新竹永久_31個」或「宜蘭系統_34個」那一層的目錄名
            # 這裡提供一個安全的寫法：紀錄檔案完整的相對路徑，這樣絕對不會漏掉資訊
            df['relative_path'] = str(file_path.relative_to(root_dir))
            
            # 欄位全轉為字串避免型態衝突
            df = df.astype(str)
            
            # 寫入總資料庫的同一張表 'sample_trees'
            df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            
            print(f"  └ 成功匯入 {len(df)} 筆樣木資料。")
            total_success_count += 1
        else:
            print(f"  ⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        print(f"  ❌ 讀取或寫入資料庫時發生錯誤: {e}")
        total_failed_count += 1
        
    print('-' * 50)

# 關閉資料庫連線
conn.close()

print("\n============================================================")
print("🎉 大資料夾全自動掃描匯入結束！")
print(f"總共成功匯入 {total_success_count} 個檔案，失敗 {total_failed_count} 個檔案。")
print(f"總資料庫檔案路徑: {db_path}")
print("============================================================")

正在建立/連接總資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\total_forest_plots_auto.db
正在自動掃描大資料夾下所有的 Excel 檔案...
Reading 113系統及永久樣區複查數量表(79個)_資料繳交統計.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Worksheet named '樣木' not found
--------------------------------------------------
Reading p_06002.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06006.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Empty table or column name specified
--------------------------------------------------
Reading p_06015.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Empty table or column name specified
--------------------------------------------------
Reading p_06019.xls ...
  └ 成功匯入 32 筆樣木資料。
--------------------------------------------------
Reading p_06023.xls ...
  └ 成功匯入 29 筆樣木資料。
--------------------------------------------------
Reading p_06026.xls ...
  └ 成功匯入 56 筆樣木資料。
--------------------------------------------------
Reading p_06028.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Read

In [4]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 🎯 這裡只需要提供最外層的大資料夾路徑即可！
main_dir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料"
sheet_name = '樣木'

# 設定總資料庫儲存路徑（直接放在大資料夾底下）
root_dir = Path(main_dir)
db_path = root_dir / "全_forest_plots_auto.db"

# 建立或連接到 SQLite 資料庫
print(f"正在建立/連接總資料庫：{db_path}")
conn = sqlite3.connect(db_path)

# 用來記錄全局成功與失敗的計數器
total_success_count = 0
total_failed_count = 0

# 🚀 使用 rglob("*.xls*") 直接對最外層大資料夾進行深度搜尋，它會自動掘進所有子目錄
print("正在自動掃描大資料夾下所有的 Excel 檔案...")
excel_files = root_dir.rglob("*.xls*")

for file_path in excel_files:
    # 跳過 Excel 的暫存檔
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} ...")
    df = None
    
    try:
        # 1. 檢查檔案開頭是否為 XML
        is_xml_spreadsheet = False
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        # 2. 根據真實格式讀取
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name and ws_name[0] == sheet_name:
                    target_worksheet = ws
                    break
            
            if target_worksheet is not None:
                rows_data = []  # 這裡改用來儲存每一列的字典 (dict)
                
                for row in target_worksheet.findall('.//Row'):
                    row_dict = {}
                    current_col = 1  # Excel XML 的欄位索引預設從 1 開始
                    
                    # 改為先尋找 Cell 標籤，而不是直接找 Data
                    for cell in row.findall('Cell'):
                        # 檢查 Cell 是否有 Index 屬性（用來處理跳格、空格）
                        idx_attr = [v for k, v in cell.attrib.items() if 'Index' in k]
                        if idx_attr:
                            current_col = int(idx_attr[0])  # 如果有指定 Index，強制跳到該欄位序號
                        
                        # 再從 Cell 之中尋找 Data 標籤
                        data_elem = cell.find('Data')
                        cell_text = data_elem.text if (data_elem is not None and data_elem.text) else ""
                        
                        # 將資料根據準確的欄位序號存入字典
                        row_dict[current_col] = cell_text
                        current_col += 1  # 預設自動往下一個欄位移動
                    
                    if row_dict:
                        rows_data.append(row_dict)
                
                if rows_data:
                    # 將字典列表轉為 DataFrame，Pandas 會非常聰明地根據 key (欄位序號) 自動對齊！
                    raw_df = pd.DataFrame(rows_data)
                    
                    # 補齊可能因末尾欄位缺失而沒產生出來的行數，並確保欄位順序從 1 開始連續排列
                    all_cols = range(1, int(raw_df.columns.max()) + 1)
                    raw_df = raw_df.reindex(columns=all_cols).fillna("")
                    
                    # 取第一列作為表頭，並清理表頭的換行符號
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
                    
        else:
            if file_path.suffix.lower() == '.xls':
                df = pd.read_excel(file_path, sheet_name=sheet_name, engine='xlrd')
            else:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')

            # 🎯 【新增這行】自動過濾掉表頭名稱包含 "Unnamed:" 的幽靈欄位
            df = df.loc[:, ~df.columns.str.contains('^Unnamed:', case=False, na=False)]
            
            # 💡 【自動智慧欄位解析】
            # 利用 file_path.parts 來取得檔案相對於大資料夾的路徑層級
            # file_path.parent.name 通常是個別的「樣區編號」或「公司代號」資料夾
            # 我們可以抓取上一層或上兩層的目錄名稱，來自動識別所屬的分署與樣區類型
            df['source_file'] = file_path.name           # 檔案名稱
            df['parent_folder'] = file_path.parent.name   # 直接上層資料夾名稱
            
            # 自動抓取路徑中代表「新竹永久_31個」或「宜蘭系統_34個」那一層的目錄名
            # 這裡提供一個安全的寫法：紀錄檔案完整的相對路徑，這樣絕對不會漏掉資訊
            df['relative_path'] = str(file_path.relative_to(root_dir))
            
            # 欄位全轉為字串避免型態衝突
            df = df.astype(str)
            
            # 寫入總資料庫的同一張表 'sample_trees'
            df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            
            print(f"  └ 成功匯入 {len(df)} 筆樣木資料。")
            total_success_count += 1
        else:
            print(f"  ⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        print(f"  ❌ 讀取或寫入資料庫時發生錯誤: {e}")
        total_failed_count += 1
        
    print('-' * 50)

# 關閉資料庫連線
conn.close()

print("\n============================================================")
print("🎉 大資料夾全自動掃描匯入結束！")
print(f"總共成功匯入 {total_success_count} 個檔案，失敗 {total_failed_count} 個檔案。")
print(f"總資料庫檔案路徑: {db_path}")
print("============================================================")

正在建立/連接總資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\全_forest_plots_auto.db
正在自動掃描大資料夾下所有的 Excel 檔案...
Reading 113系統及永久樣區複查數量表(79個)_資料繳交統計.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Worksheet named '樣木' not found
--------------------------------------------------
Reading p_06002.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06006.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Empty table or column name specified
--------------------------------------------------
Reading p_06015.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Empty table or column name specified
--------------------------------------------------
Reading p_06019.xls ...
  └ 成功匯入 32 筆樣木資料。
--------------------------------------------------
Reading p_06023.xls ...
  └ 成功匯入 29 筆樣木資料。
--------------------------------------------------
Reading p_06026.xls ...
  └ 成功匯入 56 筆樣木資料。
--------------------------------------------------
Reading p_06028.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading 

In [5]:
#僅偵測工作表樣木
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

main_dir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料"
sheet_name = '樣木'

root_dir = Path(main_dir)
db_path = root_dir / "全部_forest_plots_auto.db"

print(f"正在建立/連接總資料庫：{db_path}")
conn = sqlite3.connect(db_path)

total_success_count = 0
total_failed_count = 0

print("正在自動掃描大資料夾下所有的 Excel 檔案...")
excel_files = root_dir.rglob("*.xls*")

for file_path in excel_files:
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} ...")
    df = None
    
    try:
        is_xml_spreadsheet = False
        # 增加容錯：如果檔案被 Excel 開啟，這裡 open 就會噴 Permission denied，會直接被 try-except 捕捉並提示
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name and ws_name[0] == sheet_name:
                    target_worksheet = ws
                    break
            
            if target_worksheet is not None:
                rows_data = []
                for row in target_worksheet.findall('.//Row'):
                    row_dict = {}
                    current_col = 1
                    for cell in row.findall('Cell'):
                        idx_attr = [v for k, v in cell.attrib.items() if 'Index' in k]
                        if idx_attr:
                            current_col = int(idx_attr[0])
                        
                        data_elem = cell.find('Data')
                        cell_text = data_elem.text if (data_elem is not None and data_elem.text) else ""
                        row_dict[current_col] = cell_text
                        current_col += 1
                    if row_dict:
                        rows_data.append(row_dict)
                
                if rows_data:
                    raw_df = pd.DataFrame(rows_data)
                    all_cols = range(1, int(raw_df.columns.max()) + 1)
                    raw_df = raw_df.reindex(columns=all_cols).fillna("")
                    
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
        else:
            if file_path.suffix.lower() == '.xls':
                df = pd.read_excel(file_path, sheet_name=sheet_name, engine='xlrd')
            else:
                df = pd.read_excel(file_path, sheet_name=sheet_name)
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')
            
            # 剔除表頭包含 "Unnamed:" 的幽靈欄位
            df = df.loc[:, ~df.columns.str.contains('^Unnamed:', case=False, na=False)]
            
            # 🎯【新增】剔除完全空白或為空字串的無效欄位名稱（防止 Empty table or column name 錯誤）
            df = df.loc[:, (df.columns is not None) & (df.columns != "") & (~df.columns.isna())]
            
            if df.empty or len(df.columns) == 0:
                print(f"  ⚠️ (此檔案工作表有效欄位為空，跳過)")
                continue
            
            # 🎯【新增】統一常見的異體字表頭，避免資料庫因欄位名稱不一致而排斥
            # 格式為 {'舊名稱': '新名稱'}
            rename_dict = {
                '鋁牌編號': '鋁牌號碼',
                '樹種名稱': '樹種',
                '胸圍': '胸徑'  # 如果有需要可自行擴充
            }
            df = df.rename(columns=rename_dict)
            
            # 新增追溯欄位
            df['source_file'] = file_path.name
            df['parent_folder'] = file_path.parent.name
            df['relative_path'] = str(file_path.relative_to(root_dir))
            
            df = df.astype(str)
            
            # 🎯【智慧防噴錯寫入機制】
            # 先行檢查資料庫內是否已有該表格，並自動比對欄位。如果此 Excel 多了新欄位（如「序號」），自動擴充資料庫
            try:
                df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            except sqlite3.OperationalError as sql_err:
                # 如果噴錯原因是「資料庫缺少欄位」，就自動為資料庫新增欄位（ALTER TABLE）
                if "has no column named" in str(sql_err):
                    cursor = conn.cursor()
                    # 抓取目前的資料庫現有欄位
                    db_cols = [row[1] for row in cursor.execute("PRAGMA table_info(sample_trees);").fetchall()]
                    # 找出哪些欄位是 Excel 有但資料庫沒有的
                    for col in df.columns:
                        if col not in db_cols:
                            # 動態擴充資料庫欄位
                            cursor.execute(f'ALTER TABLE sample_trees ADD COLUMN "{col}" TEXT;')
                    conn.commit()
                    # 擴充完畢後，重新寫入一次
                    df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
                else:
                    raise sql_err
            
            print(f"  └ 成功匯入 {len(df)} 筆樣木資料。")
            total_success_count += 1
        else:
            print(f"  ⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        # 針對權限被拒絕進行更親民的提示
        if "Permission denied" in str(e):
            print(f"  ❌ 錯誤：請先關閉正在開啟的 Excel 檔案：{file_path.name}")
        else:
            print(f"  ❌ 讀取或寫入資料庫時發生錯誤: {e}")
        total_failed_count += 1
        
    print('-' * 50)

conn.close()

print("\n============================================================")
print(f"總共成功匯入 {total_success_count} 個檔案，失敗 {total_failed_count} 個檔案。")
print("============================================================")

正在建立/連接總資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\全部_forest_plots_auto.db
正在自動掃描大資料夾下所有的 Excel 檔案...
Reading 113系統及永久樣區複查數量表(79個)_資料繳交統計.xls ...
  ❌ 讀取或寫入資料庫時發生錯誤: Worksheet named '樣木' not found
--------------------------------------------------
Reading p_06002.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06006.xls ...
  └ 成功匯入 51 筆樣木資料。
--------------------------------------------------
Reading p_06015.xls ...
  └ 成功匯入 33 筆樣木資料。
--------------------------------------------------
Reading p_06019.xls ...
  └ 成功匯入 32 筆樣木資料。
--------------------------------------------------
Reading p_06023.xls ...
  └ 成功匯入 29 筆樣木資料。
--------------------------------------------------
Reading p_06026.xls ...
  └ 成功匯入 56 筆樣木資料。
--------------------------------------------------
Reading p_06028.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06030.xls ...
  └ 成功匯入 33 筆樣木資料。
--------------------------------------

In [5]:
#偵測全部工作表
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

#資料所在的資料夾
main_dir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料"
sheet_name = '樣木'

#檔名
root_dir = Path(main_dir)
db_path = root_dir / "new全部_forest_plots_auto.db"

print(f"正在建立/連接總資料庫：{db_path}")
conn = sqlite3.connect(db_path)

total_success_count = 0
total_failed_count = 0

print("正在自動掃描大資料夾下所有的 Excel 檔案...")
excel_files = root_dir.rglob("*.xls*")

for file_path in excel_files:
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} ...")
    df = None
    
    try:
        is_xml_spreadsheet = False
        # 增加容錯：如果檔案被 Excel 開啟，這裡 open 就會噴 Permission denied，會直接被 try-except 捕捉並提示
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name and ws_name[0] == sheet_name:
                    target_worksheet = ws
                    break
            
            if target_worksheet is not None:
                rows_data = []
                for row in target_worksheet.findall('.//Row'):
                    row_dict = {}
                    current_col = 1
                    for cell in row.findall('Cell'):
                        idx_attr = [v for k, v in cell.attrib.items() if 'Index' in k]
                        if idx_attr:
                            current_col = int(idx_attr[0])
                        
                        data_elem = cell.find('Data')
                        cell_text = data_elem.text if (data_elem is not None and data_elem.text) else ""
                        row_dict[current_col] = cell_text
                        current_col += 1
                    if row_dict:
                        rows_data.append(row_dict)
                
                if rows_data:
                    raw_df = pd.DataFrame(rows_data)
                    all_cols = range(1, int(raw_df.columns.max()) + 1)
                    raw_df = raw_df.reindex(columns=all_cols).fillna("")
                    
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
                    
        # --- 2. 標準 Excel 讀取邏輯 ---
        else:
            if file_path.suffix.lower() == '.xls':
                xl = pd.ExcelFile(file_path, engine='xlrd')
            else:
                xl = pd.ExcelFile(file_path)
            
            # 🎯 【超強模糊比對】將所有工作表名稱去掉空白、換行，並檢查是否包含「樣木」
            matched_sheets = []
            for s in xl.sheet_names:
                clean_s = str(s).replace(r'[\s\n\r]', '') # 去除工作表名稱中的所有空格與換行
                if '樣木' in clean_s:
                    matched_sheets.append(s) # 紀錄原始的工作表名稱
            
            if matched_sheets:
                # 優先取第一個匹配到的工作表（例如：'樣木 113' 或 '樣木113'）
                current_sheet_name = matched_sheets[0]
                
                # 讀取時不設表頭(header=None)，交給步驟 3 的動態雷達去掃描真正的表頭位置
                df = xl.parse(sheet_name=current_sheet_name, header=None)
                print(f"  -> 🔍 成功模糊匹配到工作表：【{current_sheet_name}】")
            else:
                df = None
                print(f"  ⚠️ (找不到任何名稱包含「樣木」的工作表，可用的工作表有: {xl.sheet_names})")
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')
            
            # 剔除表頭包含 "Unnamed:" 的幽靈欄位
            df = df.loc[:, ~df.columns.str.contains('^Unnamed:', case=False, na=False)]
            
            # 🎯【新增】剔除完全空白或為空字串的無效欄位名稱（防止 Empty table or column name 錯誤）
            df = df.loc[:, (df.columns is not None) & (df.columns != "") & (~df.columns.isna())]
            
            if df.empty or len(df.columns) == 0:
                print(f"  ⚠️ (此檔案工作表有效欄位為空，跳過)")
                continue
            
            # 🎯【新增】統一常見的異體字表頭，避免資料庫因欄位名稱不一致而排斥
            # 格式為 {'舊名稱': '新名稱'}
            rename_dict = {
                '鋁牌編號': '鋁牌號碼',
                '樹種名稱': '樹種',
                '胸圍': '胸徑'  # 如果有需要可自行擴充
            }
            df = df.rename(columns=rename_dict)
            
            # 新增追溯欄位
            df['source_file'] = file_path.name
            df['parent_folder'] = file_path.parent.name
            df['relative_path'] = str(file_path.relative_to(root_dir))
            
            df = df.astype(str)
            
            # 🎯【智慧防噴錯寫入機制】
            # 先行檢查資料庫內是否已有該表格，並自動比對欄位。如果此 Excel 多了新欄位（如「序號」），自動擴充資料庫
            try:
                df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            except sqlite3.OperationalError as sql_err:
                # 如果噴錯原因是「資料庫缺少欄位」，就自動為資料庫新增欄位（ALTER TABLE）
                if "has no column named" in str(sql_err):
                    cursor = conn.cursor()
                    # 抓取目前的資料庫現有欄位
                    db_cols = [row[1] for row in cursor.execute("PRAGMA table_info(sample_trees);").fetchall()]
                    # 找出哪些欄位是 Excel 有但資料庫沒有的
                    for col in df.columns:
                        if col not in db_cols:
                            # 動態擴充資料庫欄位
                            cursor.execute(f'ALTER TABLE sample_trees ADD COLUMN "{col}" TEXT;')
                    conn.commit()
                    # 擴充完畢後，重新寫入一次
                    df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
                else:
                    raise sql_err
            
            print(f"  └ 成功匯入 {len(df)} 筆樣木資料。")
            total_success_count += 1
        else:
            print(f"  ⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        # 針對權限被拒絕進行更親民的提示
        if "Permission denied" in str(e):
            print(f"  ❌ 錯誤：請先關閉正在開啟的 Excel 檔案：{file_path.name}")
        else:
            print(f"  ❌ 讀取或寫入資料庫時發生錯誤: {e}")
        total_failed_count += 1
        
    print('-' * 50)

conn.close()

print("\n============================================================")
print(f"總共成功匯入 {total_success_count} 個檔案，失敗 {total_failed_count} 個檔案。")
print("============================================================")

正在建立/連接總資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\new全部_forest_plots_auto.db
正在自動掃描大資料夾下所有的 Excel 檔案...
Reading 113系統及永久樣區複查數量表(79個)_資料繳交統計.xls ...
  ⚠️ (找不到任何名稱包含「樣木」的工作表，可用的工作表有: ['113系統&永久樣區 _文仁(79個) (2)', '系統_48', '永久_31'])
  ⚠️ (此檔案找不到「樣木」工作表或資料為空)
--------------------------------------------------
Reading p_06002.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06006.xls ...
  └ 成功匯入 51 筆樣木資料。
--------------------------------------------------
Reading p_06015.xls ...
  └ 成功匯入 33 筆樣木資料。
--------------------------------------------------
Reading p_06019.xls ...
  └ 成功匯入 32 筆樣木資料。
--------------------------------------------------
Reading p_06023.xls ...
  └ 成功匯入 29 筆樣木資料。
--------------------------------------------------
Reading p_06026.xls ...
  └ 成功匯入 56 筆樣木資料。
--------------------------------------------------
Reading p_06028.xls ...
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06030.xls ..

In [1]:
#偵測全部工作表(更新)
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET
import sqlite3
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

#資料所在的資料夾
main_dir = r"C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料"
sheet_name = '樣木'

#檔名
root_dir = Path(main_dir)
db_path = root_dir / "new all_forest_plots_auto.db"

print(f"正在建立/連接總資料庫：{db_path}")
conn = sqlite3.connect(db_path)

total_success_count = 0
total_failed_count = 0

print("正在自動掃描大資料夾下所有的 Excel 檔案...")
excel_files = root_dir.rglob("*.xls*")

for file_path in excel_files:
    if file_path.name.startswith("~$"):
        continue
        
    print(f"Reading {file_path.name} ...")
    df = None
    
    try:
        is_xml_spreadsheet = False
        # 增加容錯：如果檔案被 Excel 開啟，這裡 open 就會噴 Permission denied，會直接被 try-except 捕捉並提示
        with open(file_path, "rb") as f:
            start_bytes = f.read(150)
            if b"<?xml" in start_bytes:
                is_xml_spreadsheet = True
        
        # --- 1. XML 試算表讀取邏輯 ---
        if is_xml_spreadsheet:
            tree = ET.parse(file_path)
            root = tree.getroot()
            
            for elem in root.iter():
                if '}' in elem.tag:
                    elem.tag = elem.tag.split('}', 1)[1]
            
            target_worksheet = None
            for ws in root.findall('Worksheet'):
                # 抓取 Worksheet 的 Name 屬性
                ws_name = [v for k, v in ws.attrib.items() if 'Name' in k]
                if ws_name:
                    original_name = str(ws_name[0])
                    # 🎯【核心修正】徹底蒸發 XML 工作表名稱中的所有空格與換行雜訊
                    clean_name = original_name.replace(" ", "").replace("\n", "").replace("\r", "").replace("\t", "")
                    
                    if '樣木' in clean_name:
                        target_worksheet = ws
                        current_sheet_name = original_name  # 記錄原始名稱（例如：樣木 113）
                        print(f"  -> 🔍 [XML] 成功模糊匹配到工作表：【{current_sheet_name}】")
                        break
            
            if target_worksheet is not None:
                rows_data = []
                for row in target_worksheet.findall('.//Row'):
                    row_dict = {}
                    current_col = 1
                    for cell in row.findall('Cell'):
                        idx_attr = [v for k, v in cell.attrib.items() if 'Index' in k]
                        if idx_attr:
                            current_col = int(idx_attr[0])
                        
                        data_elem = cell.find('Data')
                        cell_text = data_elem.text if (data_elem is not None and data_elem.text) else ""
                        row_dict[current_col] = cell_text
                        current_col += 1
                    if row_dict:
                        rows_data.append(row_dict)
                
                if rows_data:
                    raw_df = pd.DataFrame(rows_data)
                    all_cols = range(1, int(raw_df.columns.max()) + 1)
                    raw_df = raw_df.reindex(columns=all_cols).fillna("")
                    
                    header = raw_df.iloc[0].astype(str).str.replace(r'[\s\n\r]', '', regex=True)
                    df = raw_df[1:]
                    df.columns = header
                    
        # --- 2. 標準 Excel 讀取邏輯 ---
        else:
            if file_path.suffix.lower() == '.xls':
                xl = pd.ExcelFile(file_path, engine='xlrd')
            else:
                xl = pd.ExcelFile(file_path)
            
            # 🎯 【超強模糊比對】將所有工作表名稱去掉空白、換行，並檢查是否包含「樣木」
            matched_sheets = []
            for s in xl.sheet_names:
                clean_s = str(s).replace(r'[\s\n\r]', '') # 去除工作表名稱中的所有空格與換行
                if '樣木' in clean_s:
                    matched_sheets.append(s) # 紀錄原始的工作表名稱
            
            if matched_sheets:
                # 優先取第一個匹配到的工作表（例如：'樣木 113' 或 '樣木113'）
                current_sheet_name = matched_sheets[0]
                
                # 讀取時不設表頭(header=None)，交給步驟 3 的動態雷達去掃描真正的表頭位置
                df = xl.parse(sheet_name=current_sheet_name, header=None)
                print(f"  -> 🔍 成功模糊匹配到工作表：【{current_sheet_name}】")
            else:
                df = None
                print(f"  ⚠️ (找不到任何名稱包含「樣木」的工作表，可用的工作表有: {xl.sheet_names})")
            
            if df is not None and not df.empty:
                df.columns = df.columns.astype(str).str.replace(r'[\s\n\r]', '', regex=True)
        
        # 3. 清洗資料並寫入資料庫
        if df is not None and not df.empty:
            df = df.dropna(how='all')
            
            # 剔除表頭包含 "Unnamed:" 的幽靈欄位
            df = df.loc[:, ~df.columns.str.contains('^Unnamed:', case=False, na=False)]
            
            # 🎯【新增】剔除完全空白或為空字串的無效欄位名稱（防止 Empty table or column name 錯誤）
            df = df.loc[:, (df.columns is not None) & (df.columns != "") & (~df.columns.isna())]
            
            if df.empty or len(df.columns) == 0:
                print(f"  ⚠️ (此檔案工作表有效欄位為空，跳過)")
                continue
            
            # 🎯【新增】統一常見的異體字表頭，避免資料庫因欄位名稱不一致而排斥
            # 格式為 {'舊名稱': '新名稱'}
            rename_dict = {
                '鋁牌編號': '鋁牌號碼',
                '樹種名稱': '樹種',
                '胸圍': '胸徑'  # 如果有需要可自行擴充
            }
            df = df.rename(columns=rename_dict)
            
            # 新增追溯欄位
            df['source_file'] = file_path.name
            df['parent_folder'] = file_path.parent.name
            df['relative_path'] = str(file_path.relative_to(root_dir))
            
            df = df.astype(str)
            
            # 🎯【智慧防噴錯寫入機制】
            # 先行檢查資料庫內是否已有該表格，並自動比對欄位。如果此 Excel 多了新欄位（如「序號」），自動擴充資料庫
            try:
                df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
            except sqlite3.OperationalError as sql_err:
                # 如果噴錯原因是「資料庫缺少欄位」，就自動為資料庫新增欄位（ALTER TABLE）
                if "has no column named" in str(sql_err):
                    cursor = conn.cursor()
                    # 抓取目前的資料庫現有欄位
                    db_cols = [row[1] for row in cursor.execute("PRAGMA table_info(sample_trees);").fetchall()]
                    # 找出哪些欄位是 Excel 有但資料庫沒有的
                    for col in df.columns:
                        if col not in db_cols:
                            # 動態擴充資料庫欄位
                            cursor.execute(f'ALTER TABLE sample_trees ADD COLUMN "{col}" TEXT;')
                    conn.commit()
                    # 擴充完畢後，重新寫入一次
                    df.to_sql(name='sample_trees', con=conn, if_exists='append', index=False)
                else:
                    raise sql_err
            
            print(f"  └ 成功匯入 {len(df)} 筆樣木資料。")
            total_success_count += 1
        else:
            print(f"  ⚠️ (此檔案找不到「{sheet_name}」工作表或資料為空)")
            
    except Exception as e:
        # 針對權限被拒絕進行更親民的提示
        if "Permission denied" in str(e):
            print(f"  ❌ 錯誤：請先關閉正在開啟的 Excel 檔案：{file_path.name}")
        else:
            print(f"  ❌ 讀取或寫入資料庫時發生錯誤: {e}")
        total_failed_count += 1
        
    print('-' * 50)

conn.close()

print("\n============================================================")
print(f"總共成功匯入 {total_success_count} 個檔案，失敗 {total_failed_count} 個檔案。")
print("============================================================")

正在建立/連接總資料庫：C:\Users\a0909\OneDrive\Desktop\新增資料夾\113_新竹與宜蘭_樣區資料\new all_forest_plots_auto.db
正在自動掃描大資料夾下所有的 Excel 檔案...
Reading 113系統及永久樣區複查數量表(79個)_資料繳交統計.xls ...
  ⚠️ (找不到任何名稱包含「樣木」的工作表，可用的工作表有: ['113系統&永久樣區 _文仁(79個) (2)', '系統_48', '永久_31'])
  ⚠️ (此檔案找不到「樣木」工作表或資料為空)
--------------------------------------------------
Reading p_06002.xls ...
  -> 🔍 [XML] 成功模糊匹配到工作表：【樣木】
  └ 成功匯入 46 筆樣木資料。
--------------------------------------------------
Reading p_06006.xls ...
  -> 🔍 [XML] 成功模糊匹配到工作表：【樣木】
  └ 成功匯入 51 筆樣木資料。
--------------------------------------------------
Reading p_06015.xls ...
  -> 🔍 [XML] 成功模糊匹配到工作表：【樣木】
  └ 成功匯入 33 筆樣木資料。
--------------------------------------------------
Reading p_06019.xls ...
  -> 🔍 [XML] 成功模糊匹配到工作表：【樣木】
  └ 成功匯入 32 筆樣木資料。
--------------------------------------------------
Reading p_06023.xls ...
  -> 🔍 [XML] 成功模糊匹配到工作表：【樣木】
  └ 成功匯入 29 筆樣木資料。
--------------------------------------------------
Reading p_06026.xls ...
  -> 🔍 [XML] 成功模糊匹配到工作表：【樣木】
  └ 成功匯入 5